### Classification of male and female class using captured eye images


In [141]:
# Block No: 1

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import *
from tensorflow.keras.applications.imagenet_utils import decode_predictions

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

!pip install -q umap-learn
import umap

# -------------------------------------------------
# BASE PATH
# -------------------------------------------------
BASE_PATH = "/content/drive/MyDrive/4-2/Deep Learning/Assignments/Assignment-1/Dataset"

DATASET_PATHS = {
    "flower": os.path.join(BASE_PATH, "flower"),
    "face": os.path.join(BASE_PATH, "face")
}

# -------------------------------------------------
# RESULT PATH (SAME FOR ALL MODELS)
# -------------------------------------------------
RESULT_PATH = "/content/drive/MyDrive/4-2/Deep Learning/Assignments/Assignment-1/results/eye_classification_result"

os.makedirs(RESULT_PATH, exist_ok=True)

In [142]:
# Block No: 2

def get_model_config(model_name):
    models_dict = {
        "VGG16": (VGG16, tf.keras.applications.vgg16.preprocess_input, (224, 224)),
        "VGG19": (VGG19, tf.keras.applications.vgg19.preprocess_input, (224, 224)),
        "ResNet50": (ResNet50, tf.keras.applications.resnet50.preprocess_input, (224, 224)),
        "MobileNet": (MobileNet, tf.keras.applications.mobilenet.preprocess_input, (224, 224)),
        "MobileNetV2": (MobileNetV2, tf.keras.applications.mobilenet_v2.preprocess_input, (224, 224)),
        "DenseNet121": (DenseNet121, tf.keras.applications.densenet.preprocess_input, (224, 224)),
        "InceptionV3": (InceptionV3, tf.keras.applications.inception_v3.preprocess_input, (299, 299)),
        "Xception": (Xception, tf.keras.applications.xception.preprocess_input, (299, 299)),
        "EfficientNetB0": (EfficientNetB0, tf.keras.applications.efficientnet.preprocess_input, (224, 224))
    }

    return models_dict[model_name]

In [143]:
# Block 3

def get_image_paths_and_labels(dataset_name, folder_path):

    image_paths = []
    labels = []

    if dataset_name == "flower":
        # -----------------------------
        # FLOWER: label from filename
        # -----------------------------
        for file in os.listdir(folder_path):
            if file.lower().endswith((".jpg", ".jpeg", ".png")):

                image_paths.append(os.path.join(folder_path, file))

                label = file.split("-")[0]   # class1-1 → class1
                labels.append(label)

    else:
        # -----------------------------
        # FACE: label from folder name
        # -----------------------------
        for class_name in os.listdir(folder_path):
            class_dir = os.path.join(folder_path, class_name)

            if not os.path.isdir(class_dir):
                continue

            for file in os.listdir(class_dir):
                if file.lower().endswith((".jpg", ".jpeg", ".png")):

                    image_paths.append(os.path.join(class_dir, file))
                    labels.append(class_name)   # male / female

    return image_paths, labels

In [144]:
# Block No: 4

def load_and_preprocess(img_path, preprocess_fn, img_size):
    img = image.load_img(img_path, target_size=img_size)
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_fn(x)
    return x, img

In [145]:
# Block No: 5

def predict_and_visualize(model, preprocess_fn, model_name,
                          image_paths, labels, img_size):

    fig, axes = plt.subplots(1, min(5, len(image_paths)), figsize=(20, 5))

    for idx in range(min(5, len(image_paths))):
        img_path = image_paths[idx]
        true_label = labels[idx]

        x, img = load_and_preprocess(img_path, preprocess_fn, img_size)

        preds = model.predict(x, verbose=0)
        decoded = decode_predictions(preds, top=5)[0]

        top1 = decoded[0][1]

        axes[idx].imshow(img)
        axes[idx].axis("off")
        axes[idx].set_title(f"True: {true_label}\nPred: {top1}")

        print(f"\n[{model_name}] {os.path.basename(img_path)}")
        for i, p in enumerate(decoded):
            print(f"Top-{i+1}: {p[1]} ({p[2]:.4f})")

    save_path = os.path.join(RESULT_PATH, f"{model_name}_result.png")
    plt.savefig(save_path)
    plt.close()

    print(f"\nSaved classification figure → {save_path}")

In [146]:
def extract_features(model, preprocess_fn, image_paths, labels, img_size):

    features = []
    clean_labels = []

    for i, path in enumerate(image_paths):

        x, _ = load_and_preprocess(path, preprocess_fn, img_size)

        feat = model.predict(x, verbose=0)
        features.append(feat)


        clean_labels.append(labels[i])

    features = np.vstack(features)
    clean_labels = np.array(clean_labels)

    print("Feature shape:", features.shape)
    print("Unique labels:", np.unique(clean_labels))  # DEBUG

    return features, clean_labels

In [147]:
# Block No: 7

def reduce_dimensions(features):
    pca = PCA(n_components=2)
    pca_res = pca.fit_transform(features)

    tsne = TSNE(n_components=2, perplexity=5, random_state=42)
    tsne_res = tsne.fit_transform(features)

    umap_res = umap.UMAP().fit_transform(features)

    return pca_res, tsne_res, umap_res

In [148]:
# Block No: 8

def plot_embeddings(embeddings, labels, title, filename):

    plt.figure(figsize=(6, 5))

    for lab in np.unique(labels):
        idxs = labels == lab
        plt.scatter(embeddings[idxs, 0],
                    embeddings[idxs, 1],
                    label=lab)

    plt.legend()
    plt.title(title)

    save_path = os.path.join(RESULT_PATH, filename)
    plt.savefig(save_path)
    plt.close()

    print(f"Saved → {save_path}")

In [149]:
# Block No: 9

def run_pipeline(model_name, dataset_name="flower"):

    print(f"\n==============================")
    print(f"Running Model: {model_name}")
    print(f"Dataset: {dataset_name}")
    print(f"==============================\n")

    model_class, preprocess_fn, img_size = get_model_config(model_name)

    model = model_class(weights="imagenet")

    folder = DATASET_PATHS[dataset_name]

    image_paths, labels = get_image_paths_and_labels(dataset_name, folder)

    print(f"Total images: {len(image_paths)}")

    # Classification
    predict_and_visualize(model, preprocess_fn, model_name,
                          image_paths, labels, img_size)

    # Feature extractor
    feature_model = model_class(weights="imagenet",
                                include_top=False,
                                pooling="avg")

    features, feat_labels = extract_features(
      feature_model,
      preprocess_fn,
      image_paths,
      labels,
      img_size
)

    # Dimensionality reduction
    pca_res, tsne_res, umap_res = reduce_dimensions(features)

    # Save plots
    plot_embeddings(pca_res, feat_labels,
                    f"{model_name} PCA",
                    f"{model_name}_pca.png")

    plot_embeddings(tsne_res, feat_labels,
                    f"{model_name} t-SNE",
                    f"{model_name}_tsne.png")

    plot_embeddings(umap_res, feat_labels,
                    f"{model_name} UMAP",
                    f"{model_name}_umap.png")

In [150]:
# Block No: 10
# Test with 1 model
run_pipeline("ResNet50", "face")


Running Model: ResNet50
Dataset: face

Total images: 30

[ResNet50] female2-4.jpg
Top-1: mask (0.1332)
Top-2: cleaver (0.0852)
Top-3: neck_brace (0.0504)
Top-4: lab_coat (0.0359)
Top-5: sunscreen (0.0328)

[ResNet50] female2-2.jpg
Top-1: sunscreen (0.1971)
Top-2: mask (0.1625)
Top-3: shower_curtain (0.1338)
Top-4: shower_cap (0.0742)
Top-5: Band_Aid (0.0392)

[ResNet50] female1-5.jpg
Top-1: shower_curtain (0.3629)
Top-2: face_powder (0.0906)
Top-3: lipstick (0.0903)
Top-4: wig (0.0704)
Top-5: abaya (0.0656)

[ResNet50] female1-1.jpg
Top-1: face_powder (0.2320)
Top-2: wig (0.1437)
Top-3: nipple (0.1305)
Top-4: lipstick (0.1238)
Top-5: bath_towel (0.0409)

[ResNet50] female2-3.jpg
Top-1: shower_cap (0.1444)
Top-2: sunscreen (0.0943)
Top-3: Weimaraner (0.0441)
Top-4: mask (0.0440)
Top-5: bath_towel (0.0426)

Saved classification figure → /content/drive/MyDrive/4-2/Deep Learning/Assignments/Assignment-1/results/eye_classification_result/ResNet50_result.png
Feature shape: (30, 2048)
Unique

In [151]:
# Block No: 11
# Tests All models
def run_all_models():

    model_list = [
        "VGG16",
        "VGG19",
        "ResNet50",
        "MobileNet",
        "MobileNetV2",
        "DenseNet121",
        "InceptionV3",
        "Xception",
        "EfficientNetB0"
    ]

    for model_name in model_list:
        try:
            run_pipeline(model_name, "flower")
        except Exception as e:
            print(f"Error in {model_name}: {e}")


# run_all_models()